# tp-01b 欠損監査: どの特徴量が、どこで、なぜ欠けているか

**実行場所: ホスト PC（stg）。** 先に tp-01・tp-02 を通し、`train_table.parquet` がある状態で実行する。読むだけで、データは書き換えない。

欠損している 1 セル（レース × 馬 × 特徴量）ごとに、原因を次のどれかに分類する。上から順に最初に当てはまったものを採る。

| 原因 | 意味 | 対処 |
|---|---|---|
| **正当** | 定義上、値がありえない（初出走で過去走なし、開催初日で前日傾向なし、障害レース など） | 不要（欠損として扱う） |
| **上流の欠損** | 元になる別の特徴量が欠損しているため | 上流を見る |
| **生データ未取得** | そのレースの生データ（race_index など）が無い | スクレイピング |
| **生データ欠け** | レースの生データはあるが、その馬の行が無い | スクレイピング（取り直し） |
| **生データの値が空** | 馬の行はあるが該当項目が空／0（データ元が値を出していない） | 取得元の仕様を確認 |
| **加工で欠損** | 生データに値があるのに特徴量が欠損 → **加工・結合・型変換のエラー疑い** | コード修正 |
| **要確認** | 上のどれにも分類できない | 調査 |

「正当」は定義に基づく判断なので、**そのルールが実データと整合しているか**も併せて確認する（第 3 節）。

In [ ]:
import json, sys, time
from pathlib import Path
import polars as pl
import pandas as pd

def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")

REPO_ROOT = _find_root()
sys.path.insert(0, str(REPO_ROOT))
from src.pipeline.models import missing_audit as MA

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]          # 学習対象年（2026 年は使わない）
TABLES = REPO_ROOT / "data/page_reference/tables"       # race_*_flat.parquet
TRAIN = REPO_ROOT / "data/local/modeling/training_pipeline/features/train_table.parquet"
OUT = REPO_ROOT / "data/local/modeling/training_pipeline/eda/missing_audit"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
t0 = time.time()
train = pl.read_parquet(TRAIN)
print(f"学習表: {train.height:,} 行 x {train.width} 列 | 欠損のある列: {sum(train[c].null_count() > 0 for c in train.columns)}")

## 1. 行の突合: 出馬表の行のうち、学習表に入らなかったものは何か

In [ ]:
acct = MA.row_accounting(TABLES, YEARS, train.select("race_id", "horse_number"))
display(acct.to_pandas())
print("→『正当』は取消・除外・中止など。『生データ未取得』『要確認』が 0 でなければ調査対象。")

## 2. 欠損のある特徴量の一覧（年別の欠損率）

In [ ]:
nullable = [c for c in train.columns if train[c].null_count() > 0]
train = train.with_columns(pl.col("date").cast(pl.Utf8).str.slice(0, 4).alias("_year"))
by_year = (train.group_by("_year").agg([pl.col(c).is_null().mean().alias(c) for c in nullable]).sort("_year").to_pandas().set_index("_year").T)
by_year["全体"] = [train[c].null_count() / train.height for c in by_year.index]
display(by_year.sort_values("全体", ascending=False).style.format("{:.1%}").background_gradient(cmap="Reds", axis=None))

## 3. LayerA 系の特徴量: 欠損の原因分類

生データ（race_index / race_shutuba_past / race_result の flat テーブル）と突き合わせて分類する。
最後の表は **「正当」とした条件が実データと整合しているか**（その条件に当てはまる行が本当に欠損しているか）。欠損率が 100% に近くないルールは、正当の根拠にならない。

In [ ]:
presence = MA.raw_presence(TABLES, YEARS)
ctx = MA.build_context(train, presence)
rules = MA.feature_rules(set(ctx.columns))
summary, suspects = MA.classify(ctx, rules)
print(f"分類対象: {len(rules)} 特徴量 | 要調査の行（加工で欠損／要確認）: {suspects.height:,}")

pv = (summary.group_by("feature", "原因").agg(pl.col("件数").sum()).to_pandas().pivot(index="feature", columns="原因", values="件数").fillna(0).astype(int))
pv = pv.reindex(columns=[c for c in MA.CAUSES if c in pv.columns])
pv["欠損合計"] = pv.sum(axis=1)
pv["欠損率"] = pv["欠損合計"] / train.height
display(pv.sort_values("欠損合計", ascending=False))

In [ ]:
detail = summary.to_pandas().sort_values(["feature", "件数"], ascending=[True, False])
display(detail[["feature", "原因", "理由", "件数"]])

In [ ]:
v = MA.validate_legit_rules(ctx, rules).to_pandas()
display(v.sort_values("欠損率").style.format({"欠損率": "{:.1%}"}))
bad = v[v["欠損率"] < 0.98]
print("整合していない『正当』ルール:" , "なし" if bad.empty else f"{len(bad)} 件（上の表で欠損率が低いもの。ルールを見直す）")

### 3b. 馬場傾向の時系列チェック（先読みの有無）

`track_bias_winner_3f_same_day_prior`（当日の直前の傾向）は、**その日その場その路面で、そのレースより前に行われたレース**の勝ち馬の上り3F から作る設計。
したがって、**時刻が最初のレースには値が無いはず**。値が入っていれば、まだ行われていない後続レースの結果を使っている（先読み）。

In [ ]:
# 傾向の計算に使われる全レース(race_result の全レース)を基準にする。学習表に無いレースが先頭だと『最初のレース』を取り違える
universe = (pl.concat([pl.read_parquet(TABLES / str(y) / "race_result_flat.parquet", columns=["race_id", "date", "venue_code", "surface", "start_time"]) for y in YEARS])
            .unique(subset=["race_id"]).filter(pl.col("venue_code").cast(pl.Utf8).str.to_integer(strict=False).is_between(1, 10)))
leak_sum, leak_rows = MA.track_bias_chronology_check(train, presence["winner_l3f"], universe=universe)
display(leak_sum.to_pandas())
display(leak_rows.head(15).to_pandas())
leak_rows.write_csv(OUT / "track_bias_lookahead_races.csv")
if leak_rows.height:
    print(f"要対応: 時刻が最初なのに値が入っているレースが {leak_rows.height:,} 件。同日の後続レースの結果を使っている疑い（学習・評価の先読み）。")

## 4. 履歴特徴量（tp-02）: 必要な過去走数を実データから推定して判定

履歴特徴量は『過去走が k 走以上あれば値が出る』という形が多い。各特徴量について **欠損率が 2% 以下になる最小の過去走数 k** を推定し、k 未満の欠損を『正当』、
上り3F を使う特徴量は『過去走の上り3F 有効数が足りない』を『生データ欠け』、どちらでも説明できない欠損を『要確認』とする。
推定であって確定ではない。**『必要過去走数(推定)』が None の特徴量は、過去走数では説明できない条件付き特徴量**（例: 脚質別）なので個別に確認する。

In [ ]:
layer_a_cols = set(rules) | {c for c in train.columns if c in pl.read_parquet_schema(REPO_ROOT / "data/local/modeling/training_pipeline/layer_a.parquet")}
hist_feats = [c for c in train.columns if c not in layer_a_cols and train[c].null_count() > 0]
hsum, hthr = MA.classify_history(train, hist_feats)
hpv = (hsum.group_by("feature", "原因").agg(pl.col("件数").sum()).to_pandas().pivot(index="feature", columns="原因", values="件数").fillna(0).astype(int))
hpv = hpv.reindex(columns=[c for c in MA.CAUSES if c in hpv.columns])
hpv["欠損合計"] = hpv.sum(axis=1)
out = hpv.join(hthr.to_pandas().set_index("feature"))
display(out.sort_values("欠損合計", ascending=False))

### 4b. 年による違い（2020 年はデータ範囲の始端）

学習表は 2020 年以降のレースだけで過去走を数えるため、**2020 年は 2019 年以前の走が無く、実際には経験のある馬も『初出走』扱い**になる。
2020 年の欠損率が高いのはこのためで、馬の実力ではなくデータ範囲の影響（左打ち切り）。

In [ ]:
debut = (train.group_by("_year").agg((pl.col("horse_prior_races") == 0).mean().alias("初出走の割合"), pl.len().alias("行数")).sort("_year").to_pandas())
display(debut.style.format({"初出走の割合": "{:.1%}", "行数": "{:,}"}))

## 5. ユニーク値による識別

列ごとにユニーク値を数え、**欠損の代用値**（-1・0・空文字・"-"）や、**数値のはずの列に混じる文字列**を見つける。
『競走中止で着順が数字でない』のような例外も、ここで値として現れる。ただし**元の文字列が加工で捨てられている列は、値を見ても区別できない**
（例: 着順は 取/除/中/失 も解析失敗もすべて -1 になる）。その場合は周辺の列（タイム・通過順）からの推定になる。

In [ ]:
rr = pl.concat([pl.read_parquet(TABLES / str(y) / "race_result_flat.parquet") for y in YEARS], how="diagonal_relaxed")
cols = [c for c in ["finish_position", "bracket_number", "horse_number", "weight", "weight_change", "popularity", "odds", "time_sec", "last_3f", "margin", "passing_order", "jockey_weight", "finish_time"] if c in rr.columns]
display(MA.value_profile(rr, cols).to_pandas())

In [ ]:
fs, fdetail = MA.finish_status_evidence(rr)
display(fs.to_pandas())
print("元の着順トークン（取/除/中/失/降 か解析失敗か）はパーサが -1 にまとめて捨てている。タイムありの行は失格・降着か解析失敗のどちらか。")
display(fdetail.to_pandas().head(30))

## 6. 要調査の行（加工で欠損／要確認）と保存

In [ ]:
top = (suspects.group_by("feature", "原因", "理由").len().rename({"len": "行数"}).sort("行数", descending=True))
display(top.to_pandas())
samples = (suspects.sort("race_id").group_by("feature", "原因", maintain_order=True).head(5))
display(samples.to_pandas())

summary.write_csv(OUT / "cause_summary_layer_a.csv"); hsum.write_csv(OUT / "cause_summary_history.csv")
hthr.write_csv(OUT / "history_thresholds.csv"); suspects.write_csv(OUT / "suspects_layer_a.csv")
v_pl = MA.validate_legit_rules(ctx, rules); v_pl.write_csv(OUT / "legit_rule_validation.csv")
acct.write_csv(OUT / "row_accounting.csv"); fdetail.write_csv(OUT / "finish_status_suspects.csv")
meta = {"rows": train.height, "layer_a_features": len(rules), "history_features": len(hist_feats),
        "suspect_rows_layer_a": suspects.height, "elapsed_sec": round(time.time() - t0, 1)}
(OUT / "summary.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存:", OUT, meta)